# 06 — Scoring a city with the frozen model

Parameters: `city`, `model_version` (which model file to load), `run_label` (`first_run` for the first run on C, `after_fix` after the changes).

Does the same as 05 does for A/B but **trains nothing**: it loads the model from the volume, splits the city into train/test with the same rule,
fits the baseline on this city's training split and computes MAE on its test split. It writes to the same gold tables with its own `run_label`,
so the dashboard and Genie see city C next to A and B.

⚠️ On city C this notebook runs **only as a Job task** after bronze → silver → 04. The first run used `model_version = v1`, `run_label = first_run`;
the run after the model changes uses `model_version = v2`, `run_label = after_fix`, `dry_run = no`.

For a `v2` model the price level of the city is the median price of the city's **own training split** (the same rows the own-city baseline uses),
and density is taken relative to the city's own median density. Only the test split is evaluated.
To test it on Rome/Sicily set `dry_run = yes` — nothing is written to gold.

In [0]:
dbutils.widgets.text("city", "rome")
dbutils.widgets.text("model_version", "v2")
dbutils.widgets.text("run_label", "after_fix")
dbutils.widgets.text("model_dir", "/Volumes/workspace/gold/models")
dbutils.widgets.dropdown("dry_run", "yes", ["no", "yes"])

city = dbutils.widgets.get("city").strip().lower()
if not city:
    raise ValueError("Set city explicitly (this notebook is meant to be run by the Job, not by hand)")
DRY_RUN = dbutils.widgets.get("dry_run") == "yes"
MODEL_VERSION = dbutils.widgets.get("model_version").strip()
RUN_LABEL = dbutils.widgets.get("run_label").strip()
MODEL_DIR = dbutils.widgets.get("model_dir").rstrip("/")

In [0]:
import os, sys, datetime
import numpy as np, pandas as pd, sklearn, joblib
from pyspark.sql import functions as F

sys.path.append(os.getcwd())
import model_utils as mu

CATALOG = "workspace"
FEATURES = f"{CATALOG}.gold.listing_features"
PREDICTIONS = f"{CATALOG}.gold.price_predictions"
METRICS = f"{CATALOG}.gold.model_metrics"

art = joblib.load(f"{MODEL_DIR}/price_model_{MODEL_VERSION}.joblib")
model = art["model"]
spec = mu.spec_of(art)
print(f"Model {art['model_version']} (spec {spec}): trained {art['trained_at']} on {art['train_cities']}, sklearn {art['sklearn_version']}")
if art["sklearn_version"] != sklearn.__version__:
    print(f"⚠ sklearn is now {sklearn.__version__} but the model was saved with {art['sklearn_version']} — note this in the report")
if city in art["train_cities"]:
    print(f"⚠ {city} is a training city; report its numbers from 05 (test), not from this run")

In [0]:
pdf = spark.table(FEATURES).filter(F.col("city") == city).toPandas()
if pdf.empty:
    raise ValueError(f"No features for '{city}' — run bronze → silver → 04_gold_features")

pdf = mu.add_relative_density(pdf)              # v2 feature; relative to ALL listings of the city
pdf["in_model_scope"] = mu.in_scope(pdf)
scope = pdf[pdf["in_model_scope"]].copy()
scope["split"] = mu.host_split(scope)
pdf["split"] = scope["split"].reindex(pdf.index).fillna("out_of_scope")
train = scope[scope.split == "train"]
display(pdf.groupby("split").size().reset_index(name="rows"))

# city C baseline is fitted on C's OWN train split (what the client would have without a model); baseline_ab is the A+B rule
own_baseline = mu.fit_baseline(train)
pdf["baseline_pred"] = mu.predict_baseline(own_baseline, pdf)
pdf["baseline_ab_pred"] = mu.predict_baseline(art["baseline_ab"], pdf)
# v2: the city's price level comes from its own training split
levels = mu.fit_city_level(train) if mu.SPECS[spec]["city_level"] else None
if levels:
    print(f"Price level of {city} (median price of its train split): {np.exp(levels[city]):.1f} EUR")
pdf["model_pred"] = mu.predict_model(model, pdf, spec, levels)

In [0]:
t = pdf[pdf["split"] == "test"]
metrics = pd.DataFrame([{"city": city, **mu.metrics_rows(t)}])
display(metrics)

## What differs from A/B (to explain the gap)

Missing share and median of every feature on C vs. A+B train, plus the share of `property_type` values the model never saw (they fall into "infrequent").

In [0]:
ref = mu.add_relative_density(spark.table(FEATURES).filter(F.col("city").isin(art["train_cities"])).toPandas())
ref = ref[ref["in_model_scope"]]
num = mu.SPECS[spec]["num"] + mu.BOOL
cmp = pd.DataFrame({
    "missing_AB": ref[num].isna().mean(), "missing_C": scope[num].isna().mean(),
    "median_AB": ref[num].astype(float).median(), "median_C": scope[num].astype(float).median(),
    "mean_AB": ref[num].astype(float).mean(), "mean_C": scope[num].astype(float).mean(),
}).round(3)
display(cmp.reset_index().rename(columns={"index": "feature"}))

enc = model.named_steps["pre"].named_transformers_["cat"]
for col, cats in zip(mu.CAT, enc.categories_):
    ic = enc.infrequent_categories_[mu.CAT.index(col)]
    infreq = set(ic) if ic is not None else set()
    known = set(cats) - infreq
    share_unknown = (~scope[col].fillna("missing").isin(known)).mean()
    print(f"{col}: {share_unknown:.1%} of listings in {city} have a value the model does not know individually")
print("Median price: A+B =", ref.price.median(), f"| {city} =", scope.price.median())

In [0]:
def save(pdf_out, table, where):
    sdf = spark.createDataFrame(pdf_out)
    w = sdf.write.format("delta").option("mergeSchema", "true")
    if spark.catalog.tableExists(table):
        w.mode("overwrite").option("replaceWhere", where).saveAsTable(table)
    else:
        w.mode("overwrite").partitionBy("city").saveAsTable(table)

p = pdf[["city", "snapshot_date", "id", "host_id", "name", "listing_url", "neighbourhood_cleansed", "latitude", "longitude",
         "room_type", "property_type", "accommodates", "bedrooms", "in_model_scope", "split", "outlier_class",
         "price", "baseline_pred", "baseline_ab_pred", "model_pred"]].copy()
p["error"] = p["model_pred"] - p["price"]
p["abs_error"] = p["error"].abs()
p["pct_error"] = p["error"] / p["price"]
p["model_vs_actual"] = np.select(
    [p["price"].isna(), p["pct_error"] < -0.2, p["pct_error"] > 0.2],
    ["no actual price", "under-priced (model < actual by >20%)", "over-priced (model > actual by >20%)"],
    default="within 20%")
p["model_version"], p["run_label"] = MODEL_VERSION, RUN_LABEL
p["snapshot_date"] = pd.to_datetime(p["snapshot_date"]).dt.date
for c in ["price", "baseline_pred", "baseline_ab_pred", "model_pred", "error", "abs_error", "pct_error", "latitude",
          "longitude", "accommodates", "bedrooms"]:
    p[c] = p[c].astype("float64")
p["in_model_scope"] = p["in_model_scope"].astype(bool)
for c in ["name", "listing_url", "neighbourhood_cleansed", "room_type", "property_type", "split", "outlier_class"]:
    p[c] = p[c].astype(object).where(p[c].notna(), None)

where = f"model_version = '{MODEL_VERSION}' AND run_label = '{RUN_LABEL}' AND city = '{city}'"
if DRY_RUN:
    print("dry_run = yes: nothing written to gold")
else:
    save(p, PREDICTIONS, where)
    save(metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL, eval_set="test",
                        train_cities=",".join(art["train_cities"]), created_at=datetime.datetime.now()), METRICS, where)

## Report table: the three numbers

Baseline on C, model MAE on A/B (from 05), model MAE on C — for every run already in `model_metrics`.

In [0]:
m = spark.table(METRICS).toPandas()
if DRY_RUN:
    m = pd.concat([m, metrics.assign(model_version=MODEL_VERSION, run_label=RUN_LABEL + " (dry run)",
                                     created_at=datetime.datetime.now())], ignore_index=True)
ab = m[(m.run_label == "train_test") & (m.city == "all")].set_index("model_version")["mae_model"]
c_runs = m[m.city == city].sort_values("created_at")
report = pd.DataFrame([{
    "run": f"{r.model_version} / {r.run_label}",
    f"baseline on {city} (own train)": r.mae_baseline,
    f"baseline A+B rule on {city}": r.mae_baseline_ab,
    "model MAE on A+B (test)": ab.get(r.model_version, np.nan),
    f"model MAE on {city} (test)": r.mae_model,
} for r in c_runs.itertuples()])
display(report)